# Variante LSTM — Juan Camilo Gallardo

Entrenamiento del `CharRNN(cell_type='lstm')` para generar nombres de dinosaurios.

Reutiliza el núcleo compartido en [src/](../src/): `dataset.py`, `model.py`, `train.py`, `sample.py`.

**Hiperparámetros**: `embed_dim=32`, `hidden_dim=128`, `num_layers=1` (+ variante con `num_layers=2, dropout=0.2`), `lr=1e-3`, `batch_size=64`, `patience=5`.

Ejecuta este notebook desde la **raíz del repo** para que los imports relativos funcionen.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / 'CLAUDE.md').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('cwd:', os.getcwd())

In [ ]:
import torch
import matplotlib.pyplot as plt
import math
from Parte_1_Generador_Caracteres.src.dataset import make_dataloaders, load_names
from Parte_1_Generador_Caracteres.src.model import CharRNN
from Parte_1_Generador_Caracteres.src.sample import load_model, generate_unique

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Datos

In [ ]:
names = load_names('data/dinos.csv')
print(f'total nombres: {len(names)}')
print(f'longitud min/max: {min(map(len, names))}/{max(map(len, names))}')
print('ejemplos:', names[:5])

In [ ]:
train_loader, val_loader, vocab, max_len = make_dataloaders('data/dinos.csv', batch_size=64)
print(f'vocab_size={len(vocab)}, max_len={max_len}')
print(f'train batches={len(train_loader)}, val batches={len(val_loader)}')

## 2. Entrenamiento — LSTM 1 capa

Configuración base: `num_layers=1`, sin dropout.

In [ ]:
from argparse import Namespace
from Parte_1_Generador_Caracteres.src.train import train

args_1layer = Namespace(
    data='data/dinos.csv',
    cell='lstm',
    epochs=50,
    batch_size=64,
    embed=32,
    hidden=128,
    layers=1,
    lr=1e-3,
    patience=5,
    checkpoint='Parte_1_Generador_Caracteres/models/lstm_1layer_JuanCamiloGallardo.pt',
    run_tag='JuanCamiloGallardo_1layer',
)
history_1, best_val_1 = train(args_1layer)
print(f'\nbest val loss (1 capa): {best_val_1:.4f}  ppl={math.exp(best_val_1):.2f}')

## 3. Entrenamiento — LSTM 2 capas + dropout 0.2

LSTM suele beneficiarse de más profundidad. Comparamos ambas configuraciones.

In [ ]:
args_2layer = Namespace(
    data='data/dinos.csv',
    cell='lstm',
    epochs=50,
    batch_size=64,
    embed=32,
    hidden=128,
    layers=2,
    lr=1e-3,
    patience=5,
    checkpoint='Parte_1_Generador_Caracteres/models/lstm_2layer_JuanCamiloGallardo.pt',
    run_tag='JuanCamiloGallardo_2layer',
)
history_2, best_val_2 = train(args_2layer)
print(f'\nbest val loss (2 capas): {best_val_2:.4f}  ppl={math.exp(best_val_2):.2f}')

## 4. Comparación de configuraciones

In [ ]:
print('Configuración        | Val Loss | Perplexity')
print('---------------------|----------|----------')
print(f'LSTM 1 capa          | {best_val_1:.4f}   | {math.exp(best_val_1):.2f}')
print(f'LSTM 2 capas dropout | {best_val_2:.4f}   | {math.exp(best_val_2):.2f}')

# Seleccionar la mejor configuración
if best_val_1 <= best_val_2:
    best_val = best_val_1
    best_args = args_1layer
    best_history = history_1
    print('\n→ Ganadora: LSTM 1 capa')
else:
    best_val = best_val_2
    best_args = args_2layer
    best_history = history_2
    print('\n→ Ganadora: LSTM 2 capas')

## 5. Curvas de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for history, label, style in [
    (history_1, 'LSTM 1 capa', '-'),
    (history_2, 'LSTM 2 capas', '--'),
]:
    ep = list(range(1, len(history['train_loss']) + 1))
    axes[0].plot(ep, history['train_loss'], style, label=f'{label} train')
    axes[0].plot(ep, history['val_loss'],   style, alpha=0.6, label=f'{label} val')
    axes[1].plot(ep, [math.exp(l) for l in history['train_loss']], style, label=f'{label} train')
    axes[1].plot(ep, [math.exp(l) for l in history['val_loss']],   style, alpha=0.6, label=f'{label} val')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CrossEntropy Loss')
axes[0].set_title('LSTM — Juan Camilo Gallardo — Loss')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].set_title('LSTM — Juan Camilo Gallardo — Perplexity')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig('reports/learning_curves_lstm.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado en reports/learning_curves_lstm.png')

## 6. Métricas finales

In [ ]:
best_train = min(best_history['train_loss'])
print(f'Épocas entrenadas : {len(best_history["train_loss"])}')
print(f'Mejor train loss  : {best_train:.4f}  (ppl = {math.exp(best_train):.2f})')
print(f'Mejor val loss    : {best_val:.4f}  (ppl = {math.exp(best_val):.2f})')
print(f'Checkpoint        : {best_args.checkpoint}')

## 7. Muestreo — 5 nombres

In [ ]:
model_lstm, vocab_lstm, max_len_lstm = load_model(best_args.checkpoint, device=device)
seen = set(load_names('data/dinos.csv'))

sample_names = generate_unique(
    model_lstm, vocab_lstm, max_len_lstm,
    n=5, temperature=1.0, top_k=None, top_p=0.9,
    seen=seen, device=device,
)
print('5 nombres generados por la LSTM (temperature=1.0, top_p=0.9):')
for name in sample_names:
    print(f'  → {name}')

## 8. Barrido de parámetros de muestreo

In [ ]:
import itertools, csv

temperatures = [0.7, 1.0, 2.5, 4.0]
top_ks       = [None, 5, 10]
top_ps       = [None, 0.9, 0.95]

rows = []
for temp, tk, tp in itertools.product(temperatures, top_ks, top_ps):
    batch = generate_unique(
        model_lstm, vocab_lstm, max_len_lstm,
        n=3, temperature=temp, top_k=tk, top_p=tp,
        seen=seen, device=device,
    )
    for name in batch:
        rows.append({'temperature': temp, 'top_k': tk, 'top_p': tp, 'name': name})

out_path = Path('data/generated/names_lstm.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['temperature', 'top_k', 'top_p', 'name'])
    writer.writeheader()
    writer.writerows(rows)
print(f'Guardados {len(rows)} nombres en {out_path}')

## 9. Copiar checkpoint final con nombre estándar

In [ ]:
import shutil

dest = Path('Parte_1_Generador_Caracteres/models/lstm_JuanCamiloGallardo.pt')
shutil.copy(best_args.checkpoint, dest)
print(f'Checkpoint copiado a {dest}')
print(f'Val loss LSTM: {best_val:.4f}  (ppl={math.exp(best_val):.2f})')
print()
print('Para promover a best_model.pt (solo si gana al RNN y GRU):')
print('  shutil.copy(dest, "Parte_1_Generador_Caracteres/models/best_model.pt")')